# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a Croissant-based dataset using the `mlcroissant` library, referencing all entities by their `@id` values for robust reproducibility and traceability.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

* https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset description
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}, Version: {dataset.metadata.version}\n")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data into **record sets**, each with fields corresponding to columns. All are referenced by their unique `@id`.

In [ ]:
# List all available record sets
print("Available record sets (referenced by @id):\n")
record_set_objs = dataset.metadata.record_sets
for rs in record_set_objs:
    print(f"- @id: {rs['@id']}, name: {rs['name']}, desc: {rs.get('description','')}")

# For each record set, list its fields and columns by @id
for rs in record_set_objs:
    print(f"\n--- Record Set @id: {rs['@id']} ---")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  Field @id: {field['@id']}, name: {field['name']}, dataType: {field.get('dataType','')}, about: {field.get('description','')}")
        # List columns for the field
        columns = field.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        for col in columns:
            cname = col['@id'] if isinstance(col, dict) else col
            print(f"    Column @id: {cname}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

Below, enter the `@id` of the target record set from the previous overview (e.g. `'cr:recordSet/ProteinAbundance'`).

For demonstration, all record sets are loaded into DataFrames and stored in a dictionary `dataframes` with their `@id` as key.

In [ ]:
from collections.abc import Iterable

# List of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} rows. Sample columns: {df.columns[:5].tolist()}")
    else:
        print(f"  No records found.")

# Choose a record set for further exploration
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # Take the first loaded non-empty for demo
    print(f"\nSelected record set for analysis: {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular dataframes available.")

## 4. Exploratory Data Analysis (EDA)
Let's perform common EDA steps, referencing fields by their `@id`.

Below, replace `field_id` and `group_field_id` with the appropriate values obtained in Section 2.

In [ ]:
# Example: suppose the main_record_set_id and fields obtained are as follows (replace accordingly):
record_set_id = main_record_set_id

# Pick a numeric field id. Replace by a valid column name (which should match a field @id, or the direct column name if that's the DataFrame column header).
# We'll attempt to infer automatically if possible.
df = dataframes[record_set_id]
numeric_candidates = df.select_dtypes('number').columns.tolist()
if not numeric_candidates:
    # Try parsing columns that contain numeric data as float (sometimes dtype=object)
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            pass
    numeric_candidates = df.select_dtypes('number').columns.tolist()

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Selected numeric field: {numeric_field_id}")
    threshold = df[numeric_field_id].median()  # Use median for robust demo
    # Filter
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical/group field
    possible_groupers = df.select_dtypes(exclude='number').columns.tolist()
    if possible_groupers:
        group_field_id = possible_groupers[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No clear categorical field for grouping. Skipping groupby.")
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following cell creates sample plots for the selected numeric field and group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure plot only runs if numeric field found
if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group field present
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found to plot.")

## 6. Conclusion
In this notebook, we demonstrated reproducible data exploration using the `mlcroissant` library against a Croissant schema, referencing all record sets and fields by their `@id`.

- You explored the available record sets and fields using their `@id`s.
- Loaded record sets into DataFrames and performed basic filtering, normalization, and grouping.
- Visualized data distributions.

**For more advanced analyses, consult the documentation for mlcroissant and review all available fields and types via the Croissant metadata.**